# MS_C1 — Multi-Station XGBoost + HGB (Perfect Forecast Full)

Trains one Optuna-tuned **XGBoost** and one **HGB** model per horizon (h=1..24) for every station in `STATIONS_TO_RUN`.
Uses `weather_mode="perfect_forecast_full"` (MET_COLS + HYSPLIT shifted to t+h).

**Hyperparameter strategy:** Optuna tunes only at representative horizons h=1, 6, 12, 24. Each remaining horizon inherits params from its nearest representative. This gives a ~6× speedup with negligible quality loss, since optimal GBM hyperparameters vary smoothly across adjacent horizons.

**Checkpoint:** skips a station if `outputs/{station}/results/C1_metrics.csv` already exists.
A kernel restart resumes from the last incomplete station.

In [11]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import time
import joblib
import numpy as np
import pandas as pd
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import src.config as cfg
import src.data_loader as dl
import src.feature_engineering as fe

from src.config import ALL_STATIONS, HORIZONS, N_OPTUNA_TRIALS, RANDOM_SEED, get_station_paths
from src.models.baseline_gbm import build_optuna_objective, _make_hgb, _make_xgb
from src.evaluation import compute_metrics
from src.utils import ensure_dirs, set_seed

set_seed(RANDOM_SEED)

WEATHER_MODE = 'perfect_forecast_full'
# Override to run a subset, e.g. STATIONS_TO_RUN = ['MzWarChrosci']
STATIONS_TO_RUN = ALL_STATIONS
# Set to a small number (e.g. 5) for a quick smoke-test
N_TRIALS = 20

print(f'Stations : {STATIONS_TO_RUN}')
print(f'Weather mode : {WEATHER_MODE} | Optuna trials: {N_TRIALS}')

Stations : ['MzWarChrosci', 'MzOtwoBrzozo', 'MzWarWokalna', 'MzWarAlNiepo', 'MzLegZegrzyn', 'MzPiasPulask', 'MzWarBajkowa']
Weather mode : perfect_forecast_full | Optuna trials: 20


In [12]:
def tune_representative_horizons(train_df, model_type, weather_mode, n_trials, random_seed):
    """Run Optuna on TUNE_HORIZONS, return dict mapping every horizon to its best params."""
    TUNE_HORIZONS = [1, 6, 12, 24]
    best_params = {}
    for h in TUNE_HORIZONS:
        X_tr, y_tr = fe.build_feature_matrix(train_df, horizon=h, weather_mode=weather_mode)
        study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(seed=random_seed)
        )
        study.optimize(
            build_optuna_objective(X_tr, y_tr, model_type=model_type),
            n_trials=n_trials, show_progress_bar=False,
        )
        best_params[h] = study.best_params
        print(f'    h={h:2d} tuned | best value: {study.best_value:.4f}')

    # Assign each horizon to nearest representative
    horizon_params = {}
    for h in HORIZONS:
        nearest = min(TUNE_HORIZONS, key=lambda t: abs(t - h))
        horizon_params[h] = best_params[nearest]
    return horizon_params


wall_start = time.time()
n = len(STATIONS_TO_RUN)

for i, station in enumerate(STATIONS_TO_RUN, 1):
    paths = get_station_paths(station)
    checkpoint = paths['results'] / 'C1_metrics.csv'

    if checkpoint.exists():
        print(f'[{i}/{n}] {station} — already done, skipping')
        continue

    print(f'\n[{i}/{n}] {station} — starting ...')

    t_station = time.time()

    ensure_dirs(paths['models'], paths['figures'], paths['results'])

    # Monkeypatch TARGET so all src functions use the current station
    cfg.TARGET = station
    dl.TARGET  = station
    fe.TARGET  = station

    df = dl.load_data()
    train_df, test_df = dl.train_test_split(df)

    xgb_models = {}
    hgb_models = {}

    # ── Tune XGBoost at representative horizons, apply to all ───────
    print(f'  Tuning XGBoost at h=1,6,12,24 (n_trials={N_TRIALS}) then fitting all 24 horizons...')
    t_xgb = time.time()
    xgb_params = tune_representative_horizons(train_df, 'xgb', WEATHER_MODE, N_TRIALS, RANDOM_SEED)
    for h in HORIZONS:
        X_tr, y_tr = fe.build_feature_matrix(train_df, horizon=h, weather_mode=WEATHER_MODE)
        model = _make_xgb(xgb_params[h])
        model.fit(X_tr.values, y_tr.values)
        joblib.dump(model, paths['models'] / f'xgb_pfxf_h{h}.pkl')
        xgb_models[h] = model
    print(f'  XGBoost done in {(time.time() - t_xgb) / 60:.1f}min')

    # ── Tune HGB at representative horizons, apply to all ───────────
    print(f'  Tuning HGB at h=1,6,12,24 (n_trials={N_TRIALS}) then fitting all 24 horizons...')
    t_hgb = time.time()
    hgb_params = tune_representative_horizons(train_df, 'hgb', WEATHER_MODE, N_TRIALS, RANDOM_SEED)
    for h in HORIZONS:
        X_tr, y_tr = fe.build_feature_matrix(train_df, horizon=h, weather_mode=WEATHER_MODE)
        model = _make_hgb(hgb_params[h])
        model.fit(X_tr.values, y_tr.values)
        joblib.dump(model, paths['models'] / f'hgb_pfxf_h{h}.pkl')
        hgb_models[h] = model
    print(f'  HGB done in {(time.time() - t_hgb) / 60:.1f}min')

    # ── Evaluate on test set ─────────────────────────────────────────
    rows = []
    for h in HORIZONS:
        X_te, y_te = fe.build_feature_matrix(test_df, horizon=h, weather_mode=WEATHER_MODE)
        xgb_preds = xgb_models[h].predict(X_te.values)
        hgb_preds = hgb_models[h].predict(X_te.values)
        rows.append({'Model': 'C1_XGBoost_pfxf', 'Station': station, 'Horizon': h,
                     **compute_metrics(y_te.values, xgb_preds)})
        rows.append({'Model': 'C1_HGB_pfxf',     'Station': station, 'Horizon': h,
                     **compute_metrics(y_te.values, hgb_preds)})

    # Print key-horizon summary
    results_df = pd.DataFrame(rows)
    for model_name in ['C1_XGBoost_pfxf', 'C1_HGB_pfxf']:
        sub = results_df[results_df['Model'] == model_name]
        print(f'  {model_name}:')
        for h in [1, 6, 12, 24]:
            r = sub[sub['Horizon'] == h].iloc[0]
            print(f'    h={h:2d}: MAE={r["MAE"]:.3f}  RMSE={r["RMSE"]:.3f}  R2={r["R2"]:.3f}')

    # Save metrics — this file acts as the checkpoint
    results_df.to_csv(checkpoint, index=False)

    elapsed_min = (time.time() - t_station) / 60
    total_elapsed_min = (time.time() - wall_start) / 60
    avg_per_station = total_elapsed_min / i
    remaining_min = avg_per_station * (n - i)
    print(f'[{i}/{n}] {station} — done | elapsed {elapsed_min:.1f}min | est. remaining {remaining_min:.1f}min')

# Restore original TARGET
cfg.TARGET = 'MzWarChrosci'
dl.TARGET  = 'MzWarChrosci'
fe.TARGET  = 'MzWarChrosci'

print(f'\nAll stations complete in {(time.time() - wall_start) / 60:.1f}min total.')


07:14:39 | src.data_loader | INFO | Loading data from D:\MOJE\DATA_SCIENCE\ML_WARSAW_AQI_TOY\warsaw_aq_forecast\data\raw\FINAL_merged_PM25_1g_all_seasons.csv



[1/7] MzWarChrosci — starting ...


07:14:39 | src.data_loader | INFO | Missing values per column:
MzOtwoBrzozo    1084
MzWarWokalna    2028
MzWarAlNiepo     575
MzLegZegrzyn    1511
MzPiasPulask    1002
MzWarChrosci     416
MzWarBajkowa     636
dir_48            48
dir_24            48
07:14:39 | src.data_loader | INFO | Loaded 52608 rows, 27 columns, range 2019-01-01 00:00:00 → 2024-12-31 23:00:00
07:14:39 | src.data_loader | INFO | Train: 43824 rows (2019-01-01 00:00:00 → 2023-12-31 23:00:00)
07:14:39 | src.data_loader | INFO | Test : 8784 rows (2024-01-01 00:00:00 → 2024-12-31 23:00:00)
07:14:39 | src.feature_engineering | INFO | build_feature_matrix | horizon=1 | weather_mode=perfect_forecast_full
07:14:39 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -1 h
07:14:40 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
07:14:40 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns


  Tuning XGBoost at h=1,6,12,24 (n_trials=20) then fitting all 24 horizons...


07:14:40 | src.feature_engineering | INFO |   X: (38271, 45) | y mean=17.38, y std=12.69
07:18:42 | src.feature_engineering | INFO | build_feature_matrix | horizon=6 | weather_mode=perfect_forecast_full
07:18:42 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -6 h
07:18:42 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
07:18:42 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns


...

    h=24 tuned | best value: 6.0049


12:15:05 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
12:15:05 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns
12:15:05 | src.feature_engineering | INFO |   X: (38496, 46) | y mean=18.12, y std=15.63
12:15:18 | src.feature_engineering | INFO | build_feature_matrix | horizon=2 | weather_mode=perfect_forecast_full
12:15:18 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -2 h
12:15:18 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
12:15:18 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns
12:15:18 | src.feature_engineering | INFO |   X: (38489, 46) | y mean=18.12, y std=15.64
12:15:31 | src.feature_engineering | INFO | build_feature_matrix | horizon=3 | weather_mode=perfect_forecast_full
12:15:31 | src.feature_engineering | INFO | perfect_forecast_full: shifted

  HGB done in 56.5min


12:19:47 | src.feature_engineering | INFO | build_feature_matrix | horizon=2 | weather_mode=perfect_forecast_full
12:19:47 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -2 h
12:19:47 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
12:19:47 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns
12:19:47 | src.feature_engineering | INFO |   X: (7855, 46) | y mean=14.98, y std=10.67
12:19:47 | src.feature_engineering | INFO | build_feature_matrix | horizon=3 | weather_mode=perfect_forecast_full
12:19:47 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -3 h
12:19:47 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
12:19:47 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns
12:19:47 | src.feature_engineering | INFO 

  C1_XGBoost_pfxf:
    h= 1: MAE=1.496  RMSE=2.572  R2=0.942
    h= 6: MAE=3.633  RMSE=5.730  R2=0.709
    h=12: MAE=4.469  RMSE=6.931  R2=0.570
    h=24: MAE=4.689  RMSE=7.229  R2=0.538
  C1_HGB_pfxf:
    h= 1: MAE=1.538  RMSE=2.623  R2=0.940
    h= 6: MAE=3.692  RMSE=5.819  R2=0.700
    h=12: MAE=4.426  RMSE=6.872  R2=0.577
    h=24: MAE=4.644  RMSE=7.191  R2=0.542
[7/7] MzWarBajkowa — done | elapsed 80.5min | est. remaining 0.0min

All stations complete in 305.3min total.
